# Sanity-Check Plots (Supplemental Material)

This notebook produces the sanity-check plots shown in the supplemental
material of the paper, namely:

- **Sanity Check 1**: comparison of forecast skill across hours of the day,
  using AIFS predictions and CombiPrecip observations on a held-out test set
  including the unobserved 06h / 18h hours.
- **Sanity Check 2**: spatial autocorrelation function of CombiPrecip,
  used to motivate the patch size used in patch-based diffusion training.

## Reproducing this notebook

This notebook expects:

- **CombiPrecip observations** (`CPCH_processed_*.nc`) in `../data/`
- **AIFS forecast inputs** (`AIFS_Single_pred_*.nc`) in `../data/`
- **Generated samples** for the 6h/18h supplemental test in
  `../output/generations/UNet_lam2_ValSampleStride4_6h_18hTest/`,
  produced by `scripts/generate_super_resolution.py`

If you don't have the 6h/18h generation outputs, you can still run Sanity
Check 2 (autocorrelation) — it only needs CombiPrecip observations.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import tqdm
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from statsmodels.tsa.stattools import acf

plt.rcParams["lines.linewidth"] = 2
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.size": 12, "figure.dpi": 120})

### Loading data

We load AIFS forecasts, CombiPrecip observations, and the generated samples
from the 6h/18h supplemental test run.

In [ ]:
# -------------- full AIFS inputs --------------
AIFS_inputs = xr.open_dataset(
    "../data/AIFS_Single_pred_concat_2019_2023_all_0_24Horizons_NoNeg_andLowResPrecipWetEnough_gamma.nc"
)

In [ ]:
# -------------- New hours test  --------------
root_newHours = xr.open_dataset(
    "../output/generations/UNet_lam2_ValSampleStride4_6h_18hTest/UNet_lam2_ValSampleStride4_6h_18hTest_samples.nc"
)
bias_corrected_newHours = xr.open_dataset(
    "../output/generations/UNet_lam2_ValSampleStride4_6h_18hTest/UNet_lam2_ValSampleStride4_6h_18hTest_samples_bias_corrected.nc"
)

# some days were filtered out in the combi precip preprocessing, so use it to filter
CombiPrecip_newHours = xr.open_dataset(
    "../data/CPCH_processed_precip_2019_2023_concat_NotFlat_6hSUM_ValSampleStride4_6h_18hTest_wetEnough_gamma.nc"
)

# Normalize both to datetime64[ns]
t1 = CombiPrecip_newHours.time.values  # already datetime64[ns]
t2 = pd.to_datetime(root_newHours.time.values)  # converts strings → datetime64[ns]

# Intersection (in datetime64[ns])
common_times = np.intersect1d(t1, t2)

# Convert common_times to strings for selecting on the string-typed coordinate
common_times_str = pd.DatetimeIndex(common_times).strftime("%Y-%m-%dT%H:%M:%S").values

# Select without touching the original time format
root_newHours = root_newHours.sel(time=common_times_str)

pred_newHours = root_newHours.pred
bias_corrected_newHours = bias_corrected_newHours.sel(
    time=common_times
)  # still datetime64


# -------------- AIFS extra done for extra hours --------------
AIFS_inputs_test_extra = xr.open_dataset(
    "../data/AIFS_inputs_ValSampleStride4_6h_18hTest.nc"
)
# select times filtered by the combi precip preprocess
AIFS_inputs_new_hours = AIFS_inputs_test_extra.sel(time=pred_newHours.time)

In [ ]:
# CombiPrecip
CombiPrecip = xr.open_dataset(
    "../data/CPCH_processed_precipitation_2019_2023_concat_NotFlat_6hSUM_00_12_wetEnough_gamma_wElevation.nc"
)
CombiPrecip_newHours = xr.open_dataset(
    "../data/CPCH_processed_precip_2019_2023_concat_NotFlat_6hSUM_ValSampleStride4_6h_18hTest_wetEnough_gamma.nc"
)

In [ ]:
aifs_precip = AIFS_inputs.total_precipitation_NoNeg
combi_precip = CombiPrecip.precip6h_cumulative_capped
aifs_precip_newHours = AIFS_inputs_new_hours.total_precipitation_NoNeg
combi_precip_newHours = CombiPrecip_newHours.precip6h_cumulative_capped

## Sanity Check 1: Verify difficulty of hours during the day! 

We want to see if AIFS predictions vs obs; to check if indeed some hours during the day are more difficult to predict or not.

In [ ]:
"""
Intrinsic difficulty analysis: Raw AIFS vs Observations per 6h time window.

Goal: Measure how well raw AIFS predictions match observations for each of the
4 accumulation windows (06-12, 12-18, 18-00, 00-06), to disentangle 
"intrinsic difficulty" from "model generalization" effects.

Approach: Coarsen observations onto the AIFS grid and compute error metrics
per window and per lead time.
"""


# ===========================================================================
#  2. Coarsen observations to AIFS grid
# ===========================================================================
def coarsen_obs_to_aifs(obs_da, aifs_da):
    """
    Bilinearly interpolate high-res observations onto the AIFS lat/lon grid.
    Returns a DataArray on the AIFS grid with only shared times.
    """
    # Get AIFS grid coordinates
    aifs_lats = aifs_da.lat.values
    aifs_lons = aifs_da.lon.values

    # Align times
    common_times = np.intersect1d(obs_da.time.values, aifs_da.time.values)
    obs_sel = obs_da.sel(time=common_times)

    # Interpolate obs to AIFS grid
    obs_coarse = obs_sel.interp(lat=aifs_lats, lon=aifs_lons, method="linear")

    return obs_coarse, common_times


def label_window(hour):
    """Map the valid-time hour to its accumulation window label."""
    if hour == 12:
        return "06–12"
    elif hour == 0:
        return "18–00"
    elif hour == 6:
        return "00–06"
    elif hour == 18:
        return "12–18"
    else:
        return f"?–{hour:02d}"


# ===========================================================================
#  3. Compute per-window, per-lead metrics
# ===========================================================================
def compute_metrics_per_window_and_lead(aifs_da, obs_coarse, common_times):
    """
    For each time window and each lead, compute:
      - MSE (all pixels)
      - MSE_wet (pixels where obs > 0.1 mm)
      - Bias (mean AIFS - mean obs)
      - Spatial correlation

    Returns a list of dicts for easy DataFrame construction.
    """
    aifs_sel = aifs_da.sel(time=common_times)
    leads = aifs_sel.prediction_delta.values
    n_leads = len(leads)

    results = []

    for ti, t in enumerate(common_times):
        hour = int(np.datetime64(t, "h").astype(int) % 24)
        window = label_window(hour)

        obs_field = obs_coarse.sel(time=t).values  # (lat, lon)
        if np.all(np.isnan(obs_field)):
            continue

        for li in range(n_leads):
            aifs_field = aifs_sel.isel(time=ti, prediction_delta=li).values

            # Skip if all NaN
            if np.all(np.isnan(aifs_field)):
                continue

            # Masks
            valid = np.isfinite(obs_field) & np.isfinite(aifs_field)
            if valid.sum() < 10:
                continue

            obs_v = obs_field[valid]
            aifs_v = aifs_field[valid]

            # Clamp negatives
            obs_v = np.maximum(obs_v, 0.0)
            aifs_v = np.maximum(aifs_v, 0.0)

            # MSE all
            mse_all = np.mean((aifs_v - obs_v) ** 2)

            # MSE wet (obs > 0.1 mm)
            wet_mask = obs_v > 0.1
            if wet_mask.sum() > 5:
                mse_wet = np.mean((aifs_v[wet_mask] - obs_v[wet_mask]) ** 2)
            else:
                mse_wet = np.nan

            # Bias
            bias = np.mean(aifs_v) - np.mean(obs_v)

            # Lead in hours
            lead_h = leads[li] / np.timedelta64(1, "h")

            results.append(
                {
                    "time": t,
                    "window": window,
                    "lead_idx": li,
                    "lead_hours": lead_h,
                    "mse": mse_all,
                    "mse_wet": mse_wet,
                    "bias": bias,
                    "n_wet": int(wet_mask.sum()),
                }
            )

    return results


# ===========================================================================
#  4. Process both datasets
# ===========================================================================
print("Coarsening original-hours observations to AIFS grid...")
obs_coarse_orig, times_orig = coarsen_obs_to_aifs(combi_precip, aifs_precip)
print(f"  {len(times_orig)} common times")

print("Coarsening new-hours observations to AIFS grid...")
obs_coarse_new, times_new = coarsen_obs_to_aifs(
    combi_precip_newHours, aifs_precip_newHours
)
print(f"  {len(times_new)} common times")

print("\nComputing metrics for original hours (06-12, 18-00)...")
results_orig = compute_metrics_per_window_and_lead(
    aifs_precip, obs_coarse_orig, times_orig
)

print("Computing metrics for new hours (00-06, 12-18)...")
results_new = compute_metrics_per_window_and_lead(
    aifs_precip_newHours, obs_coarse_new, times_new
)

# Combine
df = pd.DataFrame(results_orig + results_new)
print(f"\nTotal samples: {len(df)}")
print(f"Windows found: {sorted(df['window'].unique())}")
print(f"Samples per window:\n{df.groupby('window').size()}")


# ===========================================================================
#  5. Aggregate: per-window summary (averaged over all leads)
# ===========================================================================
summary = (
    df.groupby("window")
    .agg(
        mse_mean=("mse", "mean"),
        mse_wet_mean=("mse_wet", "mean"),
        bias_mean=("bias", "mean"),
        n_samples=("mse", "count"),
    )
    .round(4)
)

# Reorder windows in chronological order
window_order = ["00–06", "06–12", "12–18", "18–00"]
summary = summary.reindex(window_order)

print("\n" + "=" * 60)
print("  RAW AIFS vs OBS — per time window (all leads averaged)")
print("=" * 60)
print(summary.to_string())


# ===========================================================================
#  6. Aggregate: per-window × per-lead-bucket
# ===========================================================================
# Bucket leads into groups for readability
def lead_bucket(h):
    if h <= 6:
        return "6h"
    elif h <= 12:
        return "12h"
    elif h <= 18:
        return "18h"
    elif h <= 24:
        return "1d"
    elif h <= 48:
        return "2d"
    elif h <= 72:
        return "3d"
    elif h <= 96:
        return "4d"
    elif h <= 120:
        return "5d"
    else:
        return "6d"


df["lead_bucket"] = df["lead_hours"].apply(lead_bucket)
lead_order = ["6h", "12h", "18h", "1d", "2d", "3d", "4d", "5d", "6d"]

pivot_mse_wet = df.pivot_table(
    values="mse_wet", index="window", columns="lead_bucket", aggfunc="mean"
)
pivot_mse_wet = pivot_mse_wet.reindex(index=window_order, columns=lead_order)

print("\n" + "=" * 60)
print("  MSE_WET by window × lead time")
print("=" * 60)
print(pivot_mse_wet.round(3).to_string())


# ===========================================================================
#  7. Plots
# ===========================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {
    "00–06": "#2196F3",
    "06–12": "#4CAF50",
    "12–18": "#FF9800",
    "18–00": "#9C27B0",
}

# --- Plot 1: MSE_wet by lead, per window ---
ax = axes[0]
for window in window_order:
    sub = df[df["window"] == window].groupby("lead_hours")["mse_wet"].mean()
    ax.plot(
        sub.index, sub.values, "o-", label=window, color=colors[window], markersize=3
    )
ax.set_xlabel("Lead time (hours)")
ax.set_ylabel("MSE (wet pixels)")
ax.set_title("Raw AIFS vs Obs — MSE_wet by lead")
ax.legend(title="Window")

# --- Plot 2: Bias by lead, per window ---
ax = axes[1]
for window in window_order:
    sub = df[df["window"] == window].groupby("lead_hours")["bias"].mean()
    ax.plot(
        sub.index, sub.values, "o-", label=window, color=colors[window], markersize=3
    )
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("Lead time (hours)")
ax.set_ylabel("Bias (AIFS − Obs, mm)")
ax.set_title("Raw AIFS vs Obs — Bias by lead")
ax.legend(title="Window")

# --- Plot 3: Bar chart summary (all leads) ---
ax = axes[2]
x = np.arange(len(window_order))
width = 0.35
mse_vals = [summary.loc[w, "mse_wet_mean"] for w in window_order]
bias_vals = [abs(summary.loc[w, "bias_mean"]) for w in window_order]
bars1 = ax.bar(
    x - width / 2,
    mse_vals,
    width,
    label="MSE_wet",
    color=[colors[w] for w in window_order],
    alpha=0.85,
)
ax2 = ax.twinx()
bars2 = ax2.bar(
    x + width / 2,
    bias_vals,
    width,
    label="|Bias|",
    color=[colors[w] for w in window_order],
    alpha=0.4,
    hatch="//",
)
ax.set_xticks(x)
ax.set_xticklabels(window_order)
ax.set_ylabel("MSE_wet")
ax2.set_ylabel("|Bias| (mm)")
ax.set_title("Intrinsic difficulty by window (all leads)")
ax.legend(loc="upper left")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.savefig("intrinsic_difficulty_by_window.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nPlot saved to: intrinsic_difficulty_by_window.png")

# ===========================================================================
#  8. Difficulty ratio (for the paper)
# ===========================================================================
print("\n" + "=" * 60)
print("  Difficulty ratio relative to easiest window")
print("=" * 60)
min_mse = summary["mse_wet_mean"].min()
for w in window_order:
    ratio = summary.loc[w, "mse_wet_mean"] / min_mse
    print(
        f"  {w}:  MSE_wet = {summary.loc[w, 'mse_wet_mean']:.3f}  "
        f"(×{ratio:.2f} vs easiest)"
    )

## Sanity Check 2: Check the AutoCorrelation (ACF)

We want to see how much is the correlation in the combiPrecip data!

In [ ]:
"""
Temporal Autocorrelation Function (ACF) of 6h-accumulated precipitation.

Goal: Show how quickly precipitation fields decorrelate temporally,
to justify that the weekly split boundary gap is sufficient.

We compute the ACF in two ways:
  (a) Spatial-mean time series → classic 1D ACF
  (b) Field-level ACF: mean over all pixels of per-pixel ACF
      (more robust, captures spatial structure)

Both are computed at:
  - 12h cadence (original hours only: the actual train/test sampling)
  - 6h cadence  (combining original + newHours for a denser series)
"""

# ===========================================================================
#  1. Load data
# ===========================================================================
# combi_precip          → (time: 3279, lat: 640, lon: 710), every 12h
# combi_precip_newHours → (time: 215,  lat: 640, lon: 710), every 12h (offset by 6h)

# If running standalone, uncomment:
# CombiPrecip = xr.open_dataset("path_to_combi.nc")
# CombiPrecip_newHours = xr.open_dataset("path_to_combi_newhours.nc")
# combi_precip = CombiPrecip.precip6h_cumulative_capped
# combi_precip_newHours = CombiPrecip_newHours.precip6h_cumulative_capped


# ===========================================================================
#  2. Build time series at 12h and 6h cadence
# ===========================================================================

# --- 12h cadence: spatial mean of original hours ---
print("Computing spatial-mean precipitation at 12h cadence...")
ts_12h = combi_precip.mean(dim=["lat", "lon"]).load()
ts_12h = ts_12h.sortby("time")

# Remove NaN
valid_12h = ts_12h.values[np.isfinite(ts_12h.values)]
print(f"  {len(valid_12h)} valid time steps at 12h cadence")

# --- 6h cadence: merge original + newHours, sort by time ---
print("Building 6h-cadence series by merging original + newHours...")
ts_new = combi_precip_newHours.mean(dim=["lat", "lon"]).load()

# Combine into a single pandas Series, drop duplicates, sort
s_orig = pd.Series(ts_12h.values, index=pd.DatetimeIndex(ts_12h.time.values))
s_new = pd.Series(ts_new.values, index=pd.DatetimeIndex(ts_new.time.values))
s_all = pd.concat([s_orig, s_new]).sort_index()
s_all = s_all[~s_all.index.duplicated(keep="first")]
s_all = s_all.dropna()

# Check actual cadence
diffs = np.diff(s_all.index.values) / np.timedelta64(1, "h")
print(f"  {len(s_all)} valid time steps after merge")
print(
    f"  Time step distribution (hours): "
    f"median={np.median(diffs):.0f}, min={np.min(diffs):.0f}, max={np.max(diffs):.0f}"
)

# For 6h ACF, only use portions where we have consecutive 6h steps
# (gaps will bias the ACF otherwise)
# We'll compute ACF on the longest contiguous 6h-spaced segment
is_6h = diffs < 7  # allow small tolerance
segments = []
start = 0
for i in range(len(is_6h)):
    if not is_6h[i]:
        if i - start >= 10:  # at least 10 consecutive steps
            segments.append((start, i + 1))
        start = i + 1
if len(is_6h) - start >= 10:
    segments.append((start, len(is_6h) + 1))

print(
    f"  Found {len(segments)} contiguous 6h segments "
    f"(lengths: {[e - s for s, e in segments[:10]]}{'...' if len(segments) > 10 else ''})"
)

# Use all contiguous segments, compute ACF per segment, then average
vals_6h = s_all.values


# ===========================================================================
#  3. Compute ACF
# ===========================================================================
MAX_LAGS_12H = 30  # 30 × 12h = 15 days
MAX_LAGS_6H = 60  # 60 × 6h  = 15 days

# --- ACF at 12h cadence ---
print("\nComputing ACF at 12h cadence...")
acf_12h, confint_12h = acf(valid_12h, nlags=MAX_LAGS_12H, alpha=0.05, fft=True)
lag_hours_12h = np.arange(MAX_LAGS_12H + 1) * 12

# --- ACF at 6h cadence (averaged over contiguous segments) ---
print("Computing ACF at 6h cadence...")
acf_6h_list = []
for s, e in segments:
    seg = vals_6h[s:e]
    if len(seg) > MAX_LAGS_6H + 1:
        a = acf(seg, nlags=MAX_LAGS_6H, fft=True)
        acf_6h_list.append(a)

if acf_6h_list:
    acf_6h_mean = np.mean(acf_6h_list, axis=0)
    lag_hours_6h = np.arange(MAX_LAGS_6H + 1) * 6
    has_6h = True
    print(f"  Averaged ACF over {len(acf_6h_list)} segments")
else:
    has_6h = False
    print(
        "  [WARN] Not enough contiguous 6h data for ACF. Only 12h ACF will be plotted."
    )

# --- Field-level ACF at 12h (per-pixel, then spatial mean) ---
print("Computing field-level ACF at 12h cadence (may take a minute)...")
# Subsample spatially for efficiency: every 10th pixel
combi_sub = combi_precip.isel(
    lat=slice(None, None, 10), lon=slice(None, None, 10)
).load()
combi_sub = combi_sub.sortby("time")

nlat, nlon = combi_sub.shape[1], combi_sub.shape[2]
acf_field = np.zeros((nlat, nlon, MAX_LAGS_12H + 1))
valid_pixels = 0

for i in range(nlat):
    for j in range(nlon):
        ts_pix = combi_sub.values[:, i, j]
        mask = np.isfinite(ts_pix)
        if mask.sum() > MAX_LAGS_12H + 10:
            acf_field[i, j, :] = acf(ts_pix[mask], nlags=MAX_LAGS_12H, fft=True)
            valid_pixels += 1

acf_field_mean = np.nanmean(acf_field[:, :, :], axis=(0, 1))
# Re-normalize since some pixels might be zero
if valid_pixels > 0:
    acf_field_mean = acf_field_mean * (nlat * nlon) / valid_pixels
print(f"  Computed over {valid_pixels} / {nlat * nlon} subsampled pixels")


# ===========================================================================
#  4. Find decorrelation times
# ===========================================================================
def find_decorrelation_time(acf_vals, lag_hours, threshold=0.2):
    """Find first lag where ACF drops below threshold."""
    below = np.where(acf_vals < threshold)[0]
    if len(below) > 0:
        return lag_hours[below[0]]
    return None


decor_12h = find_decorrelation_time(acf_12h, lag_hours_12h, 0.2)
decor_field = find_decorrelation_time(acf_field_mean, lag_hours_12h, 0.2)
if has_6h:
    decor_6h = find_decorrelation_time(acf_6h_mean, lag_hours_6h, 0.2)

print("\n" + "=" * 60)
print("  Decorrelation times (ACF < 0.2)")
print("=" * 60)
print(
    f"  Spatial-mean, 12h cadence:  {decor_12h}h"
    if decor_12h
    else "  Spatial-mean, 12h cadence:  > 15 days"
)
if has_6h:
    print(
        f"  Spatial-mean, 6h cadence:   {decor_6h}h"
        if decor_6h
        else "  Spatial-mean, 6h cadence:   > 15 days"
    )
print(
    f"  Field-level, 12h cadence:   {decor_field}h"
    if decor_field
    else "  Field-level, 12h cadence:   > 15 days"
)


# ===========================================================================
#  5. Plot
# ===========================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# ---- Left: ACF curves ----
ax = axes[0]

ax.plot(
    lag_hours_12h,
    acf_12h,
    "o-",
    color="#2196F3",
    markersize=4,
    label="Spatial-mean (12h cadence)",
)
if has_6h:
    ax.plot(
        lag_hours_6h,
        acf_6h_mean,
        "s-",
        color="#FF9800",
        markersize=3,
        label="Spatial-mean (6h cadence)",
    )
ax.plot(
    lag_hours_12h,
    acf_field_mean,
    "^-",
    color="#4CAF50",
    markersize=4,
    label="Field-level mean (12h cadence)",
)

# Significance band (approximate 95% CI for white noise)
n_eff = len(valid_12h)
ci_bound = 1.96 / np.sqrt(n_eff)
ax.axhspan(-ci_bound, ci_bound, color="gray", alpha=0.15, label="95% CI (white noise)")

# Reference lines
ax.axhline(0.2, color="red", ls="--", lw=1, alpha=0.7, label="ACF = 0.2 threshold")
ax.axhline(0, color="k", lw=0.5)

# Annotate train-test gap
# Minimum gap: 6h (accumulation windows touching)
# Sample-center gap: 12h (consecutive samples in dataset)
ax.axvline(6, color="purple", ls=":", lw=2, alpha=0.7, label="Min train–test gap (6h)")
ax.axvline(
    12, color="purple", ls="-.", lw=2, alpha=0.5, label="Sample-center gap (12h)"
)

ax.set_xlabel("Lag (hours)")
ax.set_ylabel("Autocorrelation")
ax.set_title(
    "Temporal ACF of 6h-accumulated precipitation\n(CombiPrecip over Switzerland)"
)
ax.set_xlim(-1, 180)
ax.set_ylim(-0.15, 1.05)
ax.legend(fontsize=9, loc="upper right")

# ---- Right: Zoom on first 48h ----
ax = axes[1]

ax.plot(
    lag_hours_12h,
    acf_12h,
    "o-",
    color="#2196F3",
    markersize=5,
    label="Spatial-mean (12h)",
)
if has_6h:
    ax.plot(
        lag_hours_6h,
        acf_6h_mean,
        "s-",
        color="#FF9800",
        markersize=4,
        label="Spatial-mean (6h)",
    )
ax.plot(
    lag_hours_12h,
    acf_field_mean,
    "^-",
    color="#4CAF50",
    markersize=5,
    label="Field-level (12h)",
)

ax.axhspan(-ci_bound, ci_bound, color="gray", alpha=0.15)
ax.axhline(0.2, color="red", ls="--", lw=1, alpha=0.7)
ax.axhline(0, color="k", lw=0.5)
ax.axvline(6, color="purple", ls=":", lw=2, alpha=0.7, label="Min gap (6h)")
ax.axvline(12, color="purple", ls="-.", lw=2, alpha=0.5, label="Center gap (12h)")

ax.set_xlabel("Lag (hours)")
ax.set_ylabel("Autocorrelation")
ax.set_title("Zoom: first 48 hours")
ax.set_xlim(-1, 48)
ax.set_ylim(-0.15, 1.05)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("acf_precipitation_combi.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nPlot saved to: acf_precipitation_combi.png")


# ===========================================================================
#  6. Print summary for paper
# ===========================================================================
print("\n" + "=" * 60)
print("  SUMMARY FOR PAPER")
print("=" * 60)
print(f"  Variable: 6h-accumulated precipitation (CombiPrecip)")
print(f"  Domain: Switzerland")
print(f"  Time steps: {len(valid_12h)} (12h cadence)")
if has_6h:
    print(f"               {len(s_all)} (6h cadence, merged)")
print(f"")
print(f"  Lag-1 ACF (12h): {acf_12h[1]:.3f}")
if has_6h:
    print(f"  Lag-1 ACF (6h):  {acf_6h_mean[1]:.3f}")
    print(f"  Lag-2 ACF (6h):  {acf_6h_mean[2]:.3f}  [= 12h lag]")
print(f"")
print(f"  Decorrelation time (ACF < 0.2):")
if decor_12h:
    print(f"    Spatial-mean:  {decor_12h}h ({decor_12h / 24:.1f} days)")
if has_6h and decor_6h:
    print(f"    Spatial-mean (6h): {decor_6h}h ({decor_6h / 24:.1f} days)")
if decor_field:
    print(f"    Field-level:   {decor_field}h ({decor_field / 24:.1f} days)")
print(f"")
print(f"  Weekly split minimum boundary gap: 6h (accumulation windows)")
print(f"  Weekly split sample-center gap:    12h")
if decor_12h and decor_12h <= 12:
    print(f"  → Gap ≥ decorrelation time ✓ — minimal leakage expected")
elif decor_12h and decor_12h <= 36:
    print(f"  → Gap close to decorrelation time — some residual correlation,")
    print(f"    but the neutral-terrain test (weekly on 2023 ≈ yearly on 2023)")
    print(f"    confirms no meaningful inflation of metrics.")
else:
    print(f"  → Decorrelation slower than expected; rely on neutral-terrain test.")